In [ ]:
# import transformers
# import os

# mistral_path = 'mistral-transformer-4-44-2'

# with open(mistral_path + '/modeling_mistral.py', "r") as f:
#     modeling_mistral = f.read()

# with open(mistral_path + '/configuration_mistral.py', "r") as f:
#     configuration_mistral = f.read()


# transformers_path_dir = os.path.dirname(transformers.__file__)
# model_path = os.path.join(transformers_path_dir, "models/mistral/modeling_mistral.py")
# config_path = os.path.join(transformers_path_dir, "models/mistral/configuration_mistral.py")

# with open(model_path, "w") as f:
#     f.write(modeling_mistral)

# with open(config_path, "w") as f:
#     f.write(configuration_mistral)

In [ ]:
import sys

sys.path.append('distillation')

In [3]:
from arguments import Arguments
from teacher import Teacher, TeacherBGEM3, TeacherQwen3, TeacherOutput
from student import ClassificationBertModel, STSBertModel, StudentBertModel, StudentOutput, BertEmbedding
from data_utils import ClassificationDataset, BiSTSDataset, STSDataset, DataCollator
from loss import (mse_dim_weight_loss, mse_token_weight_loss, cosine_token_weight_loss,
                  mse_token_dim_weight_loss, derivative_loss, orthogonality_loss, cosine_loss)
from utils import evaluate_classification, evaluate_sts
# from train import Trainer
from typing import Optional, Dict, Any
from transformers import AutoTokenizer, AutoModel, AutoConfig
from torch import nn
import torch
import os
import numpy as np
import random


seed=42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

In [ ]:
args = Arguments(
    train_data='refactor-data/unsup_pair_dataset.csv', # chính là merge_3_data_5k_each
    val_data='', 
    test_data='',
    num_labels=1,
    batch_size=16,
    val_batch_size=64,
    max_len=128,
    
    pad_to_multiple_of=1,
    normlized=True,
    
    knowledge_distillation=True,
    finetune_hidden_states=True,
    output_attentions=True,
   
    teach_device='cuda:1',
    student_device='cuda:0',
    num_train_epochs=5,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,

    
    orthogonal_loss_weight=10,
    hard_label_loss_weight=0.5,
    loss_type='mse',

    embedding_loss_weight=10,
    # vector_embedding_warmup_ratio=0.5,

    teacher_layers_mapping=[23,24],
    student_encoder_layers_finetuned=[5,6],
    n_encoder_finetuned=6,
    finetune_embedding=True,
    hidden_loss_weights=[1,1],
    teacher_embedding_dimension=1024,

    orthogonal=False,
    span_loss=True,
    der_loss=True,
    # 76,86 là full, 76,09 là der, 76,15 là span, 75,8 là span chỉ 1 layer kd
    span_weight_pooling=True,
    span_loss_weight=False,

    p=0.1,

    output_dir='bert-llm2vec-stsbenchmark-checkpoint',

    teacher_model='BAAI/bge-m3',
    teacher_tokenizer='BAAI/bge-m3',
    # student_model='sentence-transformers/paraphrase-TinyBERT-L6-v2',
    # student_model='sentence-transformers/all-MiniLM-L6-v2',
    # student_model='sentence-transformers/nli-bert-base',
    student_model='minilmv2/MiniLMv2-L6-H768-distilled-from-BERT-Base/MiniLM-L6-H768-distilled-from-BERT-Base',
    student_tokenizer='bert-base-uncased',

    load_teacher_tokenizer_kwargs={'padding_side': 'right'},

    hf_token=''
)
geom_loss_check = True

BERT_MODEL_TASK = ClassificationBertModel
evaluate_function = evaluate_classification

In [ ]:
load_model_kwargs = {'torch_dtype': torch.float16,
                     'quantization_config': None,
                     'device_map': args.teach_device,
                     'trust_remote_code': True,
                     'output_hidden_states': args.finetune_hidden_states,
                     'output_attentions': args.output_attentions,
                     'attn_implementation': 'eager',
                     'token' : args.hf_token}

# teacher_model = TeacherLLM2VecMistral7B(model_name = args.teacher_model, 
#                                         load_model_kwargs = load_model_kwargs,
#                                         export_hidden_state_layers=args.teacher_layers_mapping, 
#                                         sentence_mean_pooling=True, 
#                                         weight_pooling=args.span_weight_pooling, 
#                                         span_weight=args.span_loss_weight, 
#                                         sft_path=None)

teacher_model = TeacherBGEM3(model_name = args.teacher_model, 
                                        load_model_kwargs = load_model_kwargs,
                                        export_hidden_state_layers=args.teacher_layers_mapping, 
                                        sentence_mean_pooling=False, 
                                        weight_pooling=args.span_weight_pooling, 
                                        span_weight=args.span_loss_weight)

# teacher_model = None

TeacherBGEM3 loading model ...


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [6]:
import math

class StudentBertModelV2(torch.nn.Module):
    def __init__(self, model, model_path, n_encoder_finetuned, 
                 teacher_hidden_size=-1, finetune_embedding=False, orthogonal=True):
        super().__init__()
        self.model = model

        if not finetune_embedding or n_encoder_finetuned < model.get_config().num_hidden_layers:
            for k, v in self.model.get_bert_model().embeddings.named_parameters():
                v.requires_grad = False

        for layer in self.model.get_bert_model().encoder.layer[:-n_encoder_finetuned]:
            for name, param in layer.named_parameters():
                param.requires_grad = False

        trainable_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in self.model.parameters())

        print('model output_attentions:', model.get_config().output_attentions)
        print('model output_attentions:', model.get_config().output_hidden_states)
        print(f"Trainable parameters: {trainable_params:,}")
        print(f"Total parameters: {total_params:,}")
        print(f"Percentage trainable: {100 * trainable_params / total_params:.2f}%")

        self.device = self.model.device

        self.proj_hidden_layers = None

        if teacher_hidden_size > 0:
            proj_list = []
            bound = 1 / math.sqrt(teacher_hidden_size)
            for i in range(len(self.model.hidden_layer_fineturn)):
                W = nn.Parameter(torch.empty(self.model.model.config.hidden_size, teacher_hidden_size))
                if orthogonal:
                    nn.init.orthogonal_(W)
                else:
                    nn.init.uniform_(W, -bound, bound)
                    
                proj_list.append(W)
            
            self.proj_hidden_layers = nn.ParameterList(proj_list)

            self.proj_embeddings = nn.Parameter(torch.empty(self.model.get_config().hidden_size, teacher_hidden_size))
            if orthogonal:
                nn.init.orthogonal_(self.proj_embeddings)
            else:
                nn.init.uniform_(self.proj_embeddings, -bound, bound)

            hidden_weight_path = os.path.join(model_path, 'proj_hidden_layers.pt')
            if os.path.exists(hidden_weight_path):
                self.proj_hidden_layers = torch.load(hidden_weight_path, weights_only=False)
            
            if os.path.exists(os.path.join(model_path, 'proj_embeddings.pt')):
                self.proj_embeddings = torch.load(os.path.join(model_path, 'proj_embeddings.pt'),
                                                  weights_only=False)

            self.proj_hidden_layers.to(self.device)
            self.proj_embeddings = nn.Parameter(self.proj_embeddings.to(self.device))

            self.k = nn.Parameter(torch.tensor(1.0, device=self.device)) 

    def encode(self, inputs) -> StudentOutput:
        inputs = {key: value.to(self.device) for key, value in inputs.items()}

        outputs = self.model(inputs)

        if outputs.hidden_states is not None and self.proj_hidden_layers is not None:
            hidden_states = []
            for i, proj_layer in enumerate(self.proj_hidden_layers):
                # hidden_states.append(outputs.hidden_states[i] @ self.proj_hidden_layers[i])
                hidden_states.append(
                    outputs.hidden_states[i] @ self.proj_embeddings.to(outputs.hidden_states[i].dtype)
                )
                
            outputs.hidden_states = hidden_states

            # outputs.hidden_states = torch.stack(hidden_states)
            # outputs.hidden_states = self.proj_hidden_layers[-1](outputs.hidden_states)
            # outputs.hidden_states = self.proj_layer(outputs.hidden_states)

        return outputs

    def save(self, path: str):
        self.model.save(path)
        if self.proj_hidden_layers is not None:
            torch.save(self.proj_hidden_layers, os.path.join(path, 'proj_hidden_layers.pt'))

        if self.proj_embeddings is not None:
            torch.save(self.proj_embeddings, os.path.join(path, 'proj_embeddings.pt'))


In [7]:
load_bert_model_kwargs = {'device_map': args.student_device,
                          'output_hidden_states': args.finetune_hidden_states,
                          'output_attentions': args.output_attentions,
                          'attn_implementation': 'eager' if args.output_attentions else 'sdpa',
                          'num_labels': args.num_labels
                          }
bert_model_task = BertEmbedding(model_name=args.student_model,
                                  load_model_kwargs=load_bert_model_kwargs,
                                  hidden_layer_fineturn=args.student_encoder_layers_finetuned,
                                  sentence_mean_pooling=True,
                                  weight_pooling=args.span_weight_pooling, 
                                  span_weight=args.span_loss_weight)

student_model = StudentBertModelV2(bert_model_task, model_path=args.student_model,
                                 n_encoder_finetuned = args.n_encoder_finetuned,
                                 teacher_hidden_size=args.teacher_embedding_dimension,
                                 finetune_embedding=args.finetune_embedding, orthogonal=args.orthogonal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /kaggle/input/datasets/winddao/minilmv2/MiniLMv2-L6-H768-distilled-from-BERT-Base/MiniLM-L6-H768-distilled-from-BERT-Base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model output_attentions: True
model output_attentions: True
Trainable parameters: 66,955,008
Total parameters: 66,955,008
Percentage trainable: 100.00%


In [8]:
from torch.nn.functional import pad
from torch.utils.data import Dataset
import pandas as pd
import torch
from transformers import PreTrainedTokenizer
from dataclasses import dataclass

from data_utils import prepare_pooler

@dataclass
class BiDataCollator:
    student_tokenizer: PreTrainedTokenizer = None
    teacher_tokenizer: PreTrainedTokenizer = None
    do_train: bool = True
    max_len: int = 512
    pad_to_multiple_of: int = 4
    return_tensors: str = 'pt'
    padding: bool = True
    return_offsets_mapping: bool = True
    instruction: str = "Given a text, retrieve semantically similar text: "
    # instruction: str = ""


    def __call__(self, batch):
        text1s, text2s, scores = [], [], []
        for text1, text2, score in batch:
            text1s.append(self.instruction + text1)
            text2s.append(self.instruction + text2)
            scores.append(score)

        texts = text1s + text2s
        
        student_inputs = self.student_tokenizer(
            texts,
            truncation=True,
            padding=self.padding,
            max_length=self.max_len,
            return_tensors=self.return_tensors,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_offsets_mapping=self.return_offsets_mapping and self.do_train
        )

        if not self.do_train:
            return student_inputs, None, torch.tensor(labels)

        student_token_offset_mapping = student_inputs.pop('offset_mapping')

        teacher_inputs = self.teacher_tokenizer(
            texts,
            truncation=True,
            padding=self.padding,
            max_length=self.max_len,
            return_tensors=self.return_tensors,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_offsets_mapping=self.return_offsets_mapping
        )
        teacher_token_offset_mapping = teacher_inputs.pop('offset_mapping')

        student_pooler_tensor, teacher_pooler_tensor = prepare_pooler(self.student_tokenizer.padding_side,
                                                                      student_token_offset_mapping,
                                                                      student_inputs['attention_mask'],
                                                                      self.teacher_tokenizer.padding_side,
                                                                      teacher_token_offset_mapping,
                                                                      teacher_inputs['attention_mask'])

        student_inputs['pooler_safe_idx'] = student_pooler_tensor['safe_idx']
        student_inputs['pooler_mask'] = student_pooler_tensor['mask']
        teacher_inputs['pooler_safe_idx'] = teacher_pooler_tensor['safe_idx']
        teacher_inputs['pooler_mask'] = teacher_pooler_tensor['mask']

        return student_inputs, teacher_inputs, torch.tensor(scores)


In [9]:
from typing import Type
from torch.utils.data import DataLoader, Dataset
from torch import nn

def info_nce(q, k, temperature=0.07, neg_valid_mask=None):
    q = F.normalize(q, dim=-1)
    k = F.normalize(k, dim=-1)

    logits = torch.matmul(q, k.T) / temperature
    labels = torch.arange(q.size(0), device=q.device)
    loss_inbatch = F.cross_entropy(logits, labels) 
    return loss_inbatch, logits
class Trainer:
    def __init__(self, student: StudentBertModel, args: Arguments, 
                 class_dataset_type: Type[Dataset], teacher_model: Teacher = None,
                 hidden_loss_weights = [1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 8, 10]):
        super().__init__()

        self.student = student.train()
        self.teacher_model = teacher_model

        if args.loss_type == "ce":
            self.criterion = nn.CrossEntropyLoss(reduction='mean')
        elif args.loss_type == "mse":
            self.criterion = nn.MSELoss(reduction='mean')

        self.mse_loss = nn.MSELoss(reduction='mean')
        self.args = args
        self.args.p = max(args.p, 1e-5)

        self.alpha = args.hard_label_loss_weight

        self.step = 0

        sum_hidden_loss_weights = sum(hidden_loss_weights)
        self.hidden_loss_weights = [w / sum_hidden_loss_weights for w in hidden_loss_weights]

        self.student_tokenizer = AutoTokenizer.from_pretrained(args.student_tokenizer, 
                                                               **args.load_student_tokenizer_kwargs)
        self.teacher_tokenizer = AutoTokenizer.from_pretrained(args.teacher_tokenizer, 
                                                               **args.load_teacher_tokenizer_kwargs)

        self.train_loader, self.val_loader, self.test_loader = self.get_data_loader(args, class_dataset_type)

        self.total_traning_steps = len(self.train_loader) * args.num_train_epochs
        self.embedding_warmup_steps = int(self.total_traning_steps * args.vector_embedding_warmup_ratio)

        self.k = nn.Parameter(torch.tensor(1.0, device=self.student.device)) 

    def get_data_loader(self, args: Arguments, class_dataset_type: Type[Dataset]):
        train_dataset = class_dataset_type(args.train_data)

        train_collate = BiDataCollator(self.student_tokenizer, self.teacher_tokenizer,
                                    do_train=True, max_len = args.max_len,
                                    pad_to_multiple_of = args.pad_to_multiple_of,
                                    return_tensors = 'pt', padding = True)

        train_loader = DataLoader(train_dataset, batch_size=args.batch_size,
                                shuffle=True, collate_fn=train_collate)

        # val_dataset = class_dataset_type(args.val_data)

        # val_collate = DataCollator(self.student_tokenizer, self.teacher_tokenizer,
        #                             do_train=False, max_len = args.max_len,
        #                             pad_to_multiple_of = args.pad_to_multiple_of,
        #                             return_tensors = 'pt', padding = True)

        # val_loader = DataLoader(val_dataset, batch_size=args.val_batch_size, collate_fn=val_collate)

        # if args.test_data is not None and len(args.test_data) > 0:
        #     test_dataset = class_dataset_type(args.test_data)
        #     test_loader = DataLoader(test_dataset, batch_size=args.val_batch_size, collate_fn=val_collate)
        # else:
        #     test_loader = None

        return train_loader, None, None

    def get_teacher_eval(self, inputs):
        outputs = self.teacher_model.encode(inputs)
        if outputs is None:
            return None

        embeddings = outputs.embeddings
        hidden_states, attentions, span_weights, hidden_dim_weights = None, None, None, None

        if outputs.hidden_states is not None:
            hidden_states = outputs.hidden_states.to(self.student.device, non_blocking=True)
        if outputs.attentions is not None:
            attentions = [tuple(t.to(self.student.device, non_blocking=True) for t in atts)
                          for atts in outputs.attentions]
        if outputs.span_weights is not None:
            span_weights = outputs.span_weights.to(self.student.device, non_blocking=True)
        if outputs.hidden_dim_weights is not None:
            hidden_dim_weights = outputs.hidden_dim_weights.to(self.student.device, non_blocking=True)

        outputs = TeacherOutput(
            embeddings = embeddings.to(self.student.device, non_blocking=True),
            hidden_states = hidden_states,
            attentions = attentions,
            span_weights = span_weights,
            hidden_dim_weights = hidden_dim_weights
        )

        return outputs
    def relation_score_loss(self, s_hidden, t_hidden, pair_weights):
        s_hidden = F.normalize(s_hidden, dim=-1, eps=1e-5)
        t_hidden = F.normalize(t_hidden, dim=-1, eps=1e-5)

        student_scores = torch.matmul(s_hidden, s_hidden.transpose(-1, -2))  # [B, N, N]
        teacher_scores = torch.matmul(t_hidden, t_hidden.transpose(-1, -2))  # [B, N, N]

        loss = F.mse_loss(student_scores, teacher_scores, reduction="none")  # [B, N, N]
        loss = (loss * pair_weights).sum() / s_hidden.size(0)

        return loss
    def knowledge_distillation_loss(self, student_outputs: StudentOutput,
                                    teacher_outputs: TeacherOutput = None):
        kd_loss = 0
        temp_loss = torch.tensor(0)

        if teacher_outputs is not None:
            if teacher_outputs.hidden_states is not None:
                span_loss = 0
                der_loss = 0
                geom_loss = 0
                n_layer = teacher_outputs.hidden_states.size(0)
                span_weights = teacher_outputs.span_weights.squeeze(-1)
                _, B, N = span_weights.size()

                mask = span_weights[-1].bool()  # [B, N]

                span_weights = span_weights ** self.args.p
                span_weights = span_weights / span_weights.sum(-1, keepdim=True)

                pair_weights = span_weights[-1].unsqueeze(2) * span_weights[-1].unsqueeze(1)
                mask = torch.eye(N, device=pair_weights.device).bool()  # (N, N)
                pair_weights[:, mask] = 0.0
                pair_weights = pair_weights / pair_weights.sum(dim=(1, 2), keepdim=True).clamp(min=1e-5)

                
                span_weights = span_weights.unsqueeze(-1)
                if self.args.span_loss:
                    for i in range(n_layer):
                        s_hidden = student_outputs.hidden_states[i]
                        t_hidden = teacher_outputs.hidden_states[i]
                        span_w = span_weights[i]

                        state_loss = cosine_token_weight_loss(s_hidden, t_hidden, span_w)
            
                        span_loss += self.hidden_loss_weights[i] * state_loss

                        if torch.isnan(span_loss):
                            print('span_loss nan')
                        # 2. Structural relation KD
                        if geom_loss_check:
                            span_w_2d = span_w.squeeze(-1)  # [B, N]

                            pair_w = span_w_2d.unsqueeze(2) * span_w_2d.unsqueeze(1)  # [B, N, N]

                            diag_mask = torch.eye(
                                pair_w.size(-1),
                                device=pair_w.device,
                                dtype=torch.bool
                            )

                            pair_w[:, diag_mask] = 0.0
                            pair_w = pair_w / pair_w.sum(dim=(1, 2), keepdim=True).clamp(min=1e-5)

                            score_loss_i = self.relation_score_loss(s_hidden, t_hidden, pair_w)
                            geom_loss += self.hidden_loss_weights[i] * score_loss_i
                if self.args.der_loss:
                    der_loss = derivative_loss(student_outputs.hidden_states,
                                            teacher_outputs.hidden_states,
                                            teacher_outputs.span_weights) / (n_layer - 1)

                    if torch.isnan(der_loss):
                        print('der_loss nan')

                kd_loss += 10 * span_loss
                kd_loss += 10 * der_loss
                kd_loss += 100 * geom_loss

            if self.args.embedding_loss_weight > 0:
                embed_loss_w = self.args.embedding_loss_weight
                student_projed = (student_outputs.embeddings @ self.student.proj_embeddings)
                teacher_embed = teacher_outputs.embeddings
                kd_loss += embed_loss_w * self.mse_loss(student_projed, teacher_embed)
        return kd_loss, temp_loss.item()
    # def knowledge_distillation_loss(self, student_outputs: StudentOutput,
    #                                 teacher_outputs: TeacherOutput = None):
    #     kd_loss = 0
    #     temp_loss = torch.tensor(0)
    #     embeddings = student_outputs.embeddings

    #     if teacher_outputs is not None:

    #         if teacher_outputs.hidden_states is not None:
    #             span_loss, der_loss = 0, 0
    #             n_layer = teacher_outputs.hidden_states.size(0)
    #             span_weights = teacher_outputs.span_weights

    #             span_weights = span_weights.squeeze() ** self.args.p
    #             span_weights = span_weights / span_weights.sum(-1, keepdim=True)
    #             span_weights = span_weights.unsqueeze(-1)


    #             if self.args.span_loss:
    #                 for i in range(n_layer):
    #                     s_hidden = student_outputs.hidden_states[i]
        #                 t_didden = teacher_outputs.hidden_states[i]
        #                 span_w = span_weights[i]

        #                 state_loss = cosine_token_weight_loss(s_hidden, t_didden, span_w)
        #                 # state_loss = mse_token_weight_loss(s_hidden, t_didden, span_w)
            
        #                 span_loss += self.hidden_loss_weights[i] * state_loss


        #                 if torch.isnan(span_loss):
        #                     print('span_loss nan')
                
        #         if self.args.der_loss:
        #             der_loss = derivative_loss(student_outputs.hidden_states,
        #                                     teacher_outputs.hidden_states,
        #                                     teacher_outputs.span_weights) / (n_layer - 1)

        #             if torch.isnan(der_loss):
        #                 print('der_loss nan')

        #         kd_loss += 10 * (span_loss + der_loss)


        #     if self.args.embedding_loss_weight > 0:
        #         embed_loss_w = self.args.embedding_loss_weight
        #         student_projed = (embeddings @ self.student.proj_embeddings)
        #         teacher_embed = teacher_outputs.embeddings
        #         kd_loss += embed_loss_w * self.mse_loss(student_projed, teacher_embed)
        # return kd_loss, temp_loss.item()


    def compute_loss(self, student_inputs, labels,
                     teacher_outputs: TeacherOutput = None):

        student_outputs = self.student.encode(student_inputs)

        embeddings = nn.functional.normalize(student_outputs.embeddings, dim=-1)
        n_pair = embeddings.size(0) // 2
        # hard_loss = self.criterion(student_outputs.logits, labels)
        pred_scores = (F.cosine_similarity(embeddings[:n_pair], embeddings[n_pair:]) + 1) * 2.5
        hard_loss = self.mse_loss(pred_scores, labels)


        kd_loss, _t_loss_, orthogonal_loss= 0, 0, 0

        if self.args.knowledge_distillation and teacher_outputs is not None:
            kd_loss, _t_loss_ = self.knowledge_distillation_loss(student_outputs, teacher_outputs)

            if self.args.orthogonal:
                if self.args.span_loss or self.args.der_loss:
                    for W in self.student.proj_hidden_layers:
                        orthogonal_loss += orthogonality_loss(W)
                    orthogonal_loss = orthogonal_loss / len(self.student.proj_hidden_layers)

                if self.args.embedding_loss_weight > 0:
                    orthogonal_loss += orthogonality_loss(self.student.proj_embeddings)

                kd_loss += self.args.orthogonal_loss_weight * orthogonal_loss

        # loss = self.alpha * hard_loss + (1.0 - self.alpha) * kd_loss
        loss = kd_loss
        # loss = hard_loss

        self.step += 1

        return loss, hard_loss
    

In [10]:
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
import datasets
import numpy as np
import torch

def eval_cls(model, eval_loader):
    preds, labels = [], []
    device = model.device
    
    with torch.cuda.amp.autocast(dtype=torch.float16):
        with torch.no_grad():
            for batch in tqdm(eval_loader):
                input_ids1 = batch["input_ids1"].to(device)
                attn1 = batch["attention_mask1"].to(device)
                label = batch["labels"]

                out1 = model(input_ids=input_ids1, attention_mask=attn1)
                emb1 = out1.last_hidden_state[:, 0, :]
        
                preds.extend(emb1.cpu().numpy())
                labels.extend(label.numpy())
    
    return preds, labels

class ClasssifyDataset(Dataset):
    def __init__(self, file_path):
        self.dataset = pd.read_csv(file_path)

    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        return {
            "text": self.dataset.iloc[idx]['text'],
            "label": torch.tensor(self.dataset.iloc[idx]['label'], dtype=torch.long),
        }

def clf_collate_fn(batch, tokenizer, max_len=512):
    s1_list = [item["text"] for item in batch]
    labels = torch.stack([item["label"] for item in batch])

    enc1 = tokenizer(
        s1_list,
        truncation=True,
        padding=True,       # chỉ pad theo câu dài nhất trong batch
        max_length=max_len,
        return_tensors="pt"
    )

    return {
        "input_ids1": enc1["input_ids"],
        "attention_mask1": enc1["attention_mask"],
        "labels": labels,
    }


def eval_classification_task(model, path_list):
    model.eval()
    print(' ✅ eval classifier')

    for train_path, dev_path in path_list:
        print(dev_path)
        eval_dataset = ClasssifyDataset(dev_path)
        eval_loader = DataLoader(
            eval_dataset,
            batch_size=64,
            shuffle=False,
            collate_fn=lambda x: clf_collate_fn(x, tokenizer)
        )
        
        train_dataset = ClasssifyDataset(train_path)
        train_loader = DataLoader(
            train_dataset,
            batch_size=64,
            shuffle=False,
            collate_fn=lambda x: clf_collate_fn(x, tokenizer)
        )

        X_train, y_train = eval_cls(model, train_loader)
        X_test, y_test = eval_cls(model, eval_loader)

        clf = LogisticRegression(
            random_state=42,
            n_jobs=1,
            max_iter=200,
            verbose=0,
        )
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        scores = {}
        accuracy = accuracy_score(y_test, y_pred)
        scores["accuracy"] = accuracy
        f1 = f1_score(y_test, y_pred, average="macro")
        scores["f1"] = f1
        print(scores)
        
    model.train()

In [11]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from tqdm import tqdm
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, average_precision_score


tokenizer = AutoTokenizer.from_pretrained('google-bert/bert-base-uncased')

class PairDataset(Dataset):
    def __init__(self, file_path):
        self.dataset = pd.read_csv(file_path)

        cols = self.dataset.columns

        if "sentence1" in cols and "sentence2" in cols:
            self.col1, self.col2 = "sentence1", "sentence2"
        elif "premise" in cols and "hypothesis" in cols:
            self.col1, self.col2 = "premise", "hypothesis"
        else:
            raise ValueError(
                "Dataset must contain either (sentence1, sentence2) or (premise, hypothesis)"
            )

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        row = self.dataset.iloc[idx]

        return {
            "sentence1": str(row[self.col1]),
            "sentence2": str(row[self.col2]),
            "label": torch.tensor(row["label"], dtype=torch.float),
        }
        

def eval_pair(model, eval_loader):
    preds, labels = [], []
    device = model.device
    
    with torch.cuda.amp.autocast(dtype=torch.float16):
        with torch.no_grad():
            for batch in tqdm(eval_loader):
                input_ids1 = batch["input_ids1"].to(device)
                attn1 = batch["attention_mask1"].to(device)
                input_ids2 = batch["input_ids2"].to(device)
                attn2 = batch["attention_mask2"].to(device)
                label = batch["labels"]


                out1 = model(input_ids=input_ids1, attention_mask=attn1)
                out2 = model(input_ids=input_ids2, attention_mask=attn2)

                emb1 = out1.last_hidden_state[:, 0, :]
                emb2 = out2.last_hidden_state[:, 0, :]
        
                # cosine similarity
                sim = F.cosine_similarity(emb1, emb2)
                score = (sim + 1) / 2
        
                preds.extend(score.cpu().numpy())
                labels.extend(label.numpy())
    
    metric = get_metric_pair_classification(preds, labels)
    print(metric)

    return metric

def get_metric_pair_classification(scores, labels):
    best_acc, best_thr = 0, 0
    for thr in np.linspace(0, 1, 200):
        preds = (scores >= thr).astype(int)
        acc = accuracy_score(labels, preds)
        if acc > best_acc:
            best_acc, best_thr = acc, thr
    preds = (scores >= best_thr).astype(int)
    return {
        "best_threshold": best_thr,
        "accuracy": best_acc,
        "f1": f1_score(labels, preds, average="macro"),
        "precision": precision_score(labels, preds, average="macro"),
        "recall": recall_score(labels, preds, average="macro"),
        "average_precision": average_precision_score(labels, scores)
    }


def eval_pair_task(model, path_list):
    model.eval()
    print(' ✅ eval_pair_task')
    for path in path_list:
        print(path)
        eval_dataset = PairDataset(path)
        eval_loader = DataLoader(
            eval_dataset,
            batch_size=64,
            shuffle=False,
            collate_fn=lambda x: collate_fn(x, tokenizer)
        )
        eval_pair(model, eval_loader)
    model.train()

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [12]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from tqdm import tqdm
import torch.nn.functional as F
from scipy.stats import pearsonr, spearmanr

tokenizer = AutoTokenizer.from_pretrained('google-bert/bert-base-uncased')

class STSDataset(Dataset):
    def __init__(self, file_path):
        self.dataset = pd.read_csv(file_path)

    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        # instruction = "Given a text, Retrieve semantically similar text: "
        instruction=""
        return {
            "sentence1": instruction + self.dataset.iloc[idx]['sentence1'],
            "sentence2": instruction + self.dataset.iloc[idx]['sentence2'],
            "label": torch.tensor(self.dataset.iloc[idx]['score'], dtype=torch.float),
        }
        
def collate_fn(batch, tokenizer, max_len=128):
    s1_list = [item["sentence1"] for item in batch]
    s2_list = [item["sentence2"] for item in batch]
    labels = torch.stack([item["label"] for item in batch])

    enc1 = tokenizer(
        s1_list,
        truncation=True,
        padding=True,       # chỉ pad theo câu dài nhất trong batch
        max_length=max_len,
        return_tensors="pt"
    )
    enc2 = tokenizer(
        s2_list,
        truncation=True,
        padding=True,
        max_length=max_len,
        return_tensors="pt"
    )

    return {
        "input_ids1": enc1["input_ids"],
        "attention_mask1": enc1["attention_mask"],
        "input_ids2": enc2["input_ids"],
        "attention_mask2": enc2["attention_mask"],
        "labels": labels,
    }

def eval_sts(model, eval_loader):
    preds, labels = [], []
    device = model.device
    
    with torch.cuda.amp.autocast(dtype=torch.float16):
        with torch.no_grad():
            for batch in tqdm(eval_loader):
                input_ids1 = batch["input_ids1"].to(device)
                attn1 = batch["attention_mask1"].to(device)
                input_ids2 = batch["input_ids2"].to(device)
                attn2 = batch["attention_mask2"].to(device)
                label = batch["labels"]


                out1 = model(input_ids=input_ids1, attention_mask=attn1)
                out2 = model(input_ids=input_ids2, attention_mask=attn2)

                # emb1 = mean_pooling(out1, attn1)
                # emb2 = mean_pooling(out2, attn2)
                emb1 = out1.last_hidden_state[:, 0, :]
                emb2 = out2.last_hidden_state[:, 0, :]
                # emb1 = out1.last_hidden_state[:, -1, :]
                # emb2 = out2.last_hidden_state[:, -1, :]
        
                # cosine similarity
                sim = F.cosine_similarity(emb1, emb2)
                score = (sim + 1) * 2.5  # scale [-1,1] -> [0,5]
        
                preds.extend(score.cpu().numpy())
                labels.extend(label.numpy())
    
    spearman_corr, _ = spearmanr(preds, labels)
    print(f"Spearman: {spearman_corr:.4f}")

    return spearman_corr


def eval_sts_task(model, path_list):
    model.eval()
    print(' ✅ eval_sts_task')
    for path in path_list:
        print(path)
        eval_dataset = STSDataset(path)
        eval_loader = DataLoader(
            eval_dataset,
            batch_size=64,
            shuffle=False,
            collate_fn=lambda x: collate_fn(x, tokenizer)
        )
        eval_sts(model, eval_loader)
    model.train()

In [ ]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd

tokenizer = AutoTokenizer.from_pretrained('google-bert/bert-base-uncased')

class STSDataset(Dataset):
    def __init__(self, file_path):
        self.dataset = pd.read_csv(file_path)

    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        instruction = "Given a text, Retrieve semantically similar text: "
        # instruction=""
        return {
            "sentence1": instruction + self.dataset.iloc[idx]['sentence1'],
            "sentence2": instruction + self.dataset.iloc[idx]['sentence2'],
            "label": torch.tensor(self.dataset.iloc[idx]['score'], dtype=torch.float),
        }
        
def collate_fn(batch, tokenizer, max_len=128):
    s1_list = [item["sentence1"] for item in batch]
    s2_list = [item["sentence2"] for item in batch]
    labels = torch.stack([item["label"] for item in batch])

    enc1 = tokenizer(
        s1_list,
        truncation=True,
        padding=True,       # chỉ pad theo câu dài nhất trong batch
        max_length=max_len,
        return_tensors="pt"
    )
    enc2 = tokenizer(
        s2_list,
        truncation=True,
        padding=True,
        max_length=max_len,
        return_tensors="pt"
    )

    return {
        "input_ids1": enc1["input_ids"],
        "attention_mask1": enc1["attention_mask"],
        "input_ids2": enc2["input_ids"],
        "attention_mask2": enc2["attention_mask"],
        "labels": labels,
    }


stsb_eval_dataset = STSDataset('embedding-data/data/stsbenchmark/test.csv')
stsb_eval_loader = DataLoader(
    stsb_eval_dataset,
    batch_size=64,
    shuffle=False,
    collate_fn=lambda x: collate_fn(x, tokenizer)
)

sick_eval_dataset = STSDataset('embedding-data/data/SICK/test.csv')
sick_eval_loader = DataLoader(
    sick_eval_dataset,
    batch_size=64,
    shuffle=False,
    collate_fn=lambda x: collate_fn(x, tokenizer)
)

sts12_eval_dataset = STSDataset('embedding-data/data/STS12/dev.csv')
sts12_eval_loader = DataLoader(
    sts12_eval_dataset,
    batch_size=64,
    shuffle=False,
    collate_fn=lambda x: collate_fn(x, tokenizer)
)

sts13_eval_dataset = STSDataset('embedding-data/sts13.csv')
sts13_eval_loader = DataLoader(
    sts13_eval_dataset,
    batch_size=64,
    shuffle=False,
    collate_fn=lambda x: collate_fn(x, tokenizer)
)

sts14_eval_dataset = STSDataset('embedding-data/sts14.csv')
sts14_eval_loader = DataLoader(
    sts14_eval_dataset,
    batch_size=64,
    shuffle=False,
    collate_fn=lambda x: collate_fn(x, tokenizer)
)

sts15_eval_dataset = STSDataset('embedding-data/sts15.csv')
sts15_eval_loader = DataLoader(
    sts15_eval_dataset,
    batch_size=64,
    shuffle=False,
    collate_fn=lambda x: collate_fn(x, tokenizer)
)

sts16_eval_dataset = STSDataset('embedding-data/sts16.csv')
sts16_eval_loader = DataLoader(
    sts16_eval_dataset,
    batch_size=64,
    shuffle=False,
    collate_fn=lambda x: collate_fn(x, tokenizer)
)


from tqdm import tqdm
import torch.nn.functional as F
from scipy.stats import pearsonr, spearmanr

def mean_pooling(model_output, attention_mask):
    # model_output: (batch, seq_len, hidden_dim)
    token_embeddings = model_output.last_hidden_state
    # mask: (batch, seq_len, 1)
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size())
    # chỉ tính mean trên các token != PAD
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
    sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
    return sum_embeddings / sum_mask

def eval_sts(model, eval_loader):
    preds, labels = [], []
    device = model.device
    
    with torch.cuda.amp.autocast(dtype=torch.float16):
        with torch.no_grad():
            for batch in tqdm(eval_loader):
                input_ids1 = batch["input_ids1"].to(device)
                attn1 = batch["attention_mask1"].to(device)
                input_ids2 = batch["input_ids2"].to(device)
                attn2 = batch["attention_mask2"].to(device)
                label = batch["labels"]


                out1 = model(input_ids=input_ids1, attention_mask=attn1)
                out2 = model(input_ids=input_ids2, attention_mask=attn2)

                emb1 = mean_pooling(out1, attn1)
                emb2 = mean_pooling(out2, attn2)
                # emb1 = out1.last_hidden_state[:, 0, :]
                # emb2 = out2.last_hidden_state[:, 0, :]
                # emb1 = out1.last_hidden_state[:, -1, :]
                # emb2 = out2.last_hidden_state[:, -1, :]
        
                # cosine similarity
                sim = F.cosine_similarity(emb1, emb2)
                score = (sim + 1) * 2.5  # scale [-1,1] -> [0,5]
        
                preds.extend(score.cpu().numpy())
                labels.extend(label.numpy())
    
    spearman_corr, _ = spearmanr(preds, labels)
    print(f"Spearman: {spearman_corr:.4f}")

    return spearman_corr

test_cls_tasks = [('multitask-data/multi-data/banking_train.csv', 
                   'multitask-data/multi-data/banking77_test.csv'),
                  ('multitask-data/multi-data/emotion_train.csv', 
                   'multitask-data/multi-data/emotion_test.csv'), 
                  ('multitask-data/multi-data/tweet_train.csv', 
                   'multitask-data/multi-data/tweet_test.csv')]

test_sts_tasks = ['multitask-data/multi-data/sick_test.csv', 
                  'multitask-data/multi-data/sts12_test.csv', 
                  'multitask-data/multi-data/stsb_test.csv']

test_pair_tasks = ['multitask-data/multi-data/mrpc_test.csv', 
                   'multitask-data/multi-data/scitail_test.csv', 
                   'multitask-data/multi-data/wic_test.csv']

def test(model):
    model.eval()
    print("cls")
    eval_classification_task(model, test_cls_tasks)
    print("pair")
    eval_pair_task(model, test_pair_tasks)
    print("sts")
    eval_sts_task(model, test_sts_tasks)

    model.train()



eval_dataset = STSDataset(args.val_data)
eval_loader = DataLoader(
    eval_dataset,
    batch_size=64,
    shuffle=False,
    collate_fn=lambda x: collate_fn(x, tokenizer)
)

def eval(model):
    model.eval()
    print('eval')
    eval_sts(model, eval_loader)
    model.train()

In [14]:
import data_utils

# data_utils.N_SPAN = 1
data_utils.TEACHER_OFFSET = 0

In [15]:
trainer = Trainer(student_model, args, class_dataset_type = BiSTSDataset, 
                  teacher_model = teacher_model, hidden_loss_weights = args.hidden_loss_weights)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

In [16]:
from torch import optim
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
from transformers import get_scheduler
from concurrent.futures import ThreadPoolExecutor
from itertools import chain

In [17]:
train_loader = trainer.train_loader
val_loader = trainer.val_loader
test_loader = trainer.test_loader

In [ ]:
# args.num_train_epochs = 4

trainer.student.float()
trainer.student.model.float()

trainer.student.train()
trainer.student.model.train()


optimizer = optim.AdamW(trainer.student.parameters(), lr=args.learning_rate)
# optimizer.add_param_group({"params": [trainer.k], "lr": 1e-3})

num_steps = len(train_loader)
total_traning_steps = num_steps * args.num_train_epochs

scaler = GradScaler()

scheduler = get_scheduler(
    name='cosine_with_min_lr',
    optimizer=optimizer,
    num_warmup_steps=int(total_traning_steps * args.warmup_ratio),
    # num_warmup_steps=0,
    num_training_steps=total_traning_steps,
    scheduler_specific_kwargs={'min_lr': 2e-6}
)

executor = ThreadPoolExecutor(max_workers=1)

best_result = 0

# Training loop
for epoch in range(args.num_train_epochs):
    print(('\n' + '%8s' + '%14s' + '%17s' * 2) % ('epoch', 'memory', 'loss', 'student_loss'))
    p_bar = tqdm(chain(train_loader, [(None, None, None)]), total=num_steps + 1)
    loss_total = 0
    student_loss_total = 0
    step = 0

    teacher_outputs = None
    next_teacher_outputs = None

    student_inputs, teacher_inputs, labels = None, None, None
    next_student_inputs, next_teacher_inputs, next_labels= None, None, None

    for batch in p_bar:
        student_inputs, teacher_inputs, labels = (next_student_inputs, 
                                                  next_teacher_inputs, 
                                                  next_labels)
        teacher_outputs = next_teacher_outputs

        next_student_inputs, next_teacher_inputs, next_labels = batch

        if (args.knowledge_distillation 
            and trainer.teacher_model is not None 
            and next_teacher_inputs is not None):

            teacher_future = executor.submit(trainer.get_teacher_eval, next_teacher_inputs)
        else:
            teacher_future = None

        if student_inputs is None:
            if args.knowledge_distillation and trainer.teacher_model is not None:
                next_teacher_outputs = teacher_future.result()
            continue

        optimizer.zero_grad(set_to_none=True)

        labels = labels.to(trainer.student.device).float()
        with autocast():
            loss, student_loss = trainer.compute_loss(student_inputs, labels, teacher_outputs)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        scheduler.step()

        loss_total += loss.item()
        student_loss_total += student_loss.item()
        step += 1

        if teacher_future is not None:
            next_teacher_outputs = teacher_future.result()


        memory = f'{torch.cuda.memory_reserved() / 1E9:.4g}G'  # (GB)
        s = ('%8s' + '%14s' + '%17.5g' * 2) % (f'{epoch + 1}/{args.num_train_epochs}', memory,
                                                loss_total / step, student_loss_total / step)
        p_bar.set_description(s)

        if torch.isnan(loss):
            break


    trainer.student.save(args.output_dir + f'-epoch{epoch}')

executor.shutdown()

/tmp/ipykernel_57/1686284161.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()



   epoch        memory             loss     student_loss


  0%|          | 1/3219 [00:00<46:52,  1.14it/s]/tmp/ipykernel_57/1686284161.py:69: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_57/1686284161.py:76: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
     1/5        1.908G           43.771           24.921: 100%|██████████| 3219/3219 [05:08<00:00, 10.42it/s]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


   epoch        memory             loss     student_loss


...


   epoch        memory             loss     student_loss


     3/5        1.908G            19.23           24.858: 100%|██████████| 3219/3219 [05:06<00:00, 10.51it/s]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


   epoch        memory             loss     student_loss


     4/5        1.908G            17.55           24.797:  93%|█████████▎| 2982/3219 [04:43<00:22, 10.64it/s]

In [ ]:
test(trainer.student.model.model)